# Day 2 · 이론 ① 로지스틱 회귀 — 추가 실습

**공식 실습:** `4-1.ipynb`  
**추가 실습:** `4-1_extra_practice.ipynb`

교안 **3-4(실습 A) → 4-6(실습 B) → 6-5(실습 C)** 순서로 실행합니다. 이 노트북은 원본 노트북 없이 단독 실행할 수 있도록 데이터 준비와 모델 학습 코드를 포함합니다.

| 블록 | 내용 | 교안 삽입 위치 |
|---|---|---|
| 실습 A | 점수(z)가 바뀌면 확률과 예측 클래스는 어떻게 달라질까? | 3-4 |
| 실습 B | 입력 특성이 바뀌면 예측 결과는 어떻게 달라질까? | 4-6 |
| 실습 C | 최고 확률이 0.5보다 작아도 분류할 수 있을까? | 6-5 |

## 0. 공통 실습 환경 준비

교안 1-3과 동일하게 훈련 데이터에 `ss.fit()`을 실행하고, 같은 기준을 `ss.transform()`으로 적용합니다. 테스트 데이터에는 별도로 `fit()` 하지 않습니다.

In [27]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

fish = pd.read_csv('https://bit.ly/fish_csv_data')

fish_input = fish[['Weight', 'Length', 'Diagonal', 'Height', 'Width']]
fish_target = fish['Species']

train_input, test_input, train_target, test_target = train_test_split(
    fish_input, fish_target, random_state=42
)

ss = StandardScaler()
ss.fit(train_input)

train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

# 실습 A — 예측 확률이 달라도 분류 결과는 같을까?

시그모이드 함수에 서로 다른 점수 `z`를 입력하여 예측 확률과 최종 분류를 비교합니다. 별도로 분류 모델을 학습하지 않습니다.

## A-1. 점수에 따른 예측 확률과 분류 결과 확인하기

In [28]:
import numpy as np
import pandas as pd

# 서로 다른 점수 준비
z_values = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])

# 시그모이드 함수로 확률 계산
probabilities = 1 / (1 + np.exp(-z_values))

# 확률이 0.5보다 크면 양성, 그렇지 않으면 음성
predictions = np.where(probabilities > 0.5, '양성', '음성')

# 결과 비교
results_a = pd.DataFrame({
    '점수(z)': z_values,
    '양성 클래스 확률': np.round(probabilities, 4),
    '예측 클래스': predictions
})

print(results_a.to_string(index=False))

 점수(z)  양성 클래스 확률 예측 클래스
  -4.0     0.0180     음성
  -2.0     0.1192     음성
   0.0     0.5000     음성
   2.0     0.8808     양성
   4.0     0.9820     양성


## A-2. 결과 비교하기

출력된 표에서 `z=-4`와 `z=-2`, `z=2`와 `z=4`의 확률과 예측 클래스를 비교합니다.

## 실습 B — 입력 특성이 바뀌면 예측 결과는 어떻게 달라질까?

**실습 목표**

- 입력 특성의 변화가 점수 `z`에 미치는 영향 확인
- 점수와 예측 확률의 변화 비교
- 학습된 계수와 점수 변화량의 관계 이해
- 입력값과 모델 파라미터를 구분

### B-1. 도미·빙어 이진 분류 모델 준비하기

교안 4-1·4-2와 같은 방식으로 학습하고, 다중 분류 실습과 구분하기 위해 모델 이름을 `lr_binary`로 사용합니다.

In [29]:
bream_smelt_indexes = (train_target == 'Bream') | (train_target == 'Smelt')
train_bream_smelt = train_scaled[bream_smelt_indexes]
target_bream_smelt = train_target[bream_smelt_indexes]

lr_binary = LogisticRegression()
lr_binary.fit(train_bream_smelt, target_bream_smelt)

print('클래스 순서:', lr_binary.classes_)
print('특성 순서:', list(fish_input.columns))
print('무게(Weight) 계수:', round(lr_binary.coef_[0, 0], 4))
print('두 번째 샘플의 실제 클래스:', target_bream_smelt.iloc[1])

클래스 순서: ['Bream' 'Smelt']
특성 순서: ['Weight', 'Length', 'Diagonal', 'Height', 'Width']
무게(Weight) 계수: -0.4045
두 번째 샘플의 실제 클래스: Smelt


### B-2. 무게를 -1, 0, +1만큼 변경하여 세 샘플 비교하기

원래 샘플을 복사한 뒤 표준화된 무게 특성값만 변경하고, 다른 네 가지 특성은 그대로 유지합니다.

**주의:** 무게 변화량은 g이 아니라 표준화된 무게 특성값의 변화량입니다. 실제 생선을 바꾼 것이 아니라 모델의 반응을 살펴보는 가상 입력 실험입니다.

In [30]:
original = train_bream_smelt[1:2].copy()  # 교안의 두 번째 도미·빙어 샘플
weight_change = 1.0  # 이 값만 변경해 다시 실행해 보세요.

lighter = original.copy()
lighter[0, 0] -= weight_change

heavier = original.copy()
heavier[0, 0] += weight_change

comparison_input = np.vstack([lighter, original, heavier])

# 빙어에 해당하는 확률 열은 classes_에서 확인
smelt_index = list(lr_binary.classes_).index('Smelt')

z_comparison = lr_binary.decision_function(comparison_input)
proba_comparison = lr_binary.predict_proba(comparison_input)
pred_comparison = lr_binary.predict(comparison_input)

results_b = pd.DataFrame({
    '표준화 무게 변화': [-weight_change, 0.0, weight_change],
    '점수(z)': np.round(z_comparison, 3),
    '빙어 확률': np.round(proba_comparison[:, smelt_index], 3),
    '예측 클래스': pred_comparison,
})
print(results_b.to_string(index=False))

 표준화 무게 변화  점수(z)  빙어 확률 예측 클래스
      -1.0  3.975  0.982  Smelt
       0.0  3.570  0.973  Smelt
       1.0  3.166  0.960  Smelt


### B-3. 계수와 점수 변화량의 관계 직접 확인하기

다른 특성을 고정한 채 무게 특성값을 1 증가시키면 점수 `z`는 무게 계수만큼 변합니다.

In [31]:
weight_coef = lr_binary.coef_[0, 0]
z_change = z_comparison[2] - z_comparison[1]
expected_change = weight_coef * weight_change

print(f'학습된 무게 계수: {weight_coef:.4f}')
print(f'표준화 무게 +{weight_change:g}에 따른 점수 변화: {z_change:.4f}')
print(f'무게 계수 × 변화량: {expected_change:.4f}')
print('두 값이 같은가?', np.isclose(expected_change, z_change))

학습된 무게 계수: -0.4045
표준화 무게 +1에 따른 점수 변화: -0.4045
무게 계수 × 변화량: -0.4045
두 값이 같은가? True


무게 계수가 음수이므로 무게 값이 커지면 점수와 빙어 확률이 낮아지지만, 세 경우 모두 빙어 확률이 0.5보다 높아 예측 클래스는 `Smelt`로 같습니다. 바뀐 것은 모델의 계수가 아니라 입력 특성입니다.

### B-4. 학생 응용 실습

B-2 코드에서 `weight_change = 2.0`으로 변경한 뒤 **B-2와 B-3 셀을 다시 실행**합니다. `original`은 셀을 실행할 때마다 새로 복사하므로 변경값이 누적되지 않습니다. 점수 변화량은 무게 계수의 2배가 되지만, 시그모이드는 비선형 함수이므로 확률의 변화량까지 2배가 되지는 않습니다.


## 실습 C — 최고 확률이 0.5보다 작아도 분류할 수 있을까?

**실습 목표**

- 여러 클래스의 점수를 소프트맥스로 변환
- 점수 차이에 따른 확률 분포 변화 확인
- 최고 확률이 0.5 미만이어도 클래스를 선택할 수 있음을 이해
- `softmax()`와 `predict_proba()`의 관계 확인

### C-1. 7종 생선의 다중 분류 모델 준비하기

교안 5-2와 같은 방식으로 학습하고, 실습 B의 `lr_binary`를 덮어쓰지 않도록 모델 이름을 `lr_multi`로 사용합니다.

In [32]:
lr_multi = LogisticRegression(C=20, max_iter=1000)
lr_multi.fit(train_scaled, train_target)

print('클래스 순서:', lr_multi.classes_)
print('계수 배열 크기:', lr_multi.coef_.shape)
print('첫 번째 테스트 샘플의 예측:', lr_multi.predict(test_scaled[:1])[0])

# 클래스별 원본 점수 확인
original_z = lr_multi.decision_function(test_scaled[:1])[0]
print('클래스별 점수:', np.round(original_z, 2))

클래스 순서: ['Bream' 'Parkki' 'Perch' 'Pike' 'Roach' 'Smelt' 'Whitefish']
계수 배열 크기: (7, 5)
첫 번째 테스트 샘플의 예측: Perch
클래스별 점수: [-6.51  1.04  5.17 -2.76  3.34  0.35 -0.63]


### C-2. 점수 배율을 변경하여 최고 확률 비교하기

원래 점수에 `1.0, 0.5, 0.2, 0.1`을 각각 곱한 가상 점수를 소프트맥스로 변환합니다. 소프트맥스의 성질을 관찰하는 **가상 실험**이며, 학습된 모델 자체는 변경하지 않습니다.

In [33]:
scale_factors = [1.0, 0.5, 0.2, 0.1]
comparison_z = np.vstack([original_z * factor for factor in scale_factors])
comparison_proba = softmax(comparison_z, axis=1)

rows = []
for factor, proba in zip(scale_factors, comparison_proba):
    best_index = np.argmax(proba)
    rows.append({
        '점수 배율': factor,
        '예측 클래스': lr_multi.classes_[best_index],
        '최고 확률': round(float(proba[best_index]), 3),
        '확률 합': round(float(proba.sum()), 3),
    })

results_c = pd.DataFrame(rows)
print(results_c.to_string(index=False))

 점수 배율 예측 클래스  최고 확률  확률 합
   1.0  Perch  0.842   1.0
   0.5  Perch  0.590   1.0
   0.2  Perch  0.320   1.0
   0.1  Perch  0.225   1.0


### C-3. 클래스별 확률 분포 직접 확인하기

표의 열 순서는 `lr_multi.classes_`와 같고, 행별 확률 합은 1입니다. 원래 점수(1.0배)의 소프트맥스 결과는 모델의 `predict_proba()`와 같아야 합니다.

In [34]:
proba_table = pd.DataFrame(
    comparison_proba,
    columns=lr_multi.classes_,
    index=[f'{factor}배' for factor in scale_factors]
)
print(proba_table.round(3).to_string())

# 원래 점수의 소프트맥스 출력은 모델의 predict_proba()와 일치해야 함
model_proba = lr_multi.predict_proba(test_scaled[:1])[0]
print('\n원래 점수의 소프트맥스 == predict_proba():',
      np.allclose(comparison_proba[0], model_proba))

      Bream  Parkki  Perch   Pike  Roach  Smelt  Whitefish
1.0배  0.000   0.014  0.842  0.000  0.135  0.007      0.003
0.5배  0.002   0.075  0.590  0.011  0.236  0.053      0.033
0.2배  0.031   0.140  0.320  0.066  0.222  0.122      0.100
0.1배  0.070   0.149  0.225  0.102  0.188  0.139      0.126

원래 점수의 소프트맥스 == predict_proba(): True


배율이 작아지면 클래스 간 점수 차이가 줄어 확률이 고르게 퍼지지만, 점수의 순서는 바뀌지 않으므로 가장 높은 확률의 클래스는 계속 `Perch`입니다. 최고 확률이 0.5보다 작은 행에서도 예측 클래스는 정해집니다.

### C-4. 학생 응용 실습

C-2 코드에서 `scale_factors = [1.0, 0.5, 0.2, 0.1, 2.0]`으로 변경한 뒤 **C-2와 C-3 셀을 다시 실행**합니다. 점수 차이가 커지면 높은 점수의 클래스에 확률이 더 집중되는지 확인해 봅니다.